# 03. Modeling

Batch 1로 학습·검증하고 Batch 2로 테스트한다. 모델 선택은 **Valid MAPE**로만 하고 Test는 보고용이다.

In [1]:
import sys; sys.path.append('../src')
import pandas as pd
from train import run

res, best = run()
print('최종 모델:', best.feature_set, best.model)
res

Batch 1 학습 29셀 / hold-out 7셀 | Batch 2 39셀 | Batch 3 44셀
최종 모델: B RandomForest


,feature_set,model,mape_train_cv,mape_valid,mape_test_b2,mape_test_b3,gap_train_valid,gap_valid_test,gap_target_test,final
0,A,Baseline,9.14,7.80,28.56,12.81,-1.34,20.76,19.46,False
1,A,ElasticNet,9.22,9.57,30.44,12.25,0.36,20.86,21.34,False
2,A,Ridge,9.10,7.82,28.79,12.72,-1.28,20.96,19.69,False
3,A,PLS,9.14,7.80,28.56,12.81,-1.34,20.76,19.46,False
4,A,GPR,9.71,8.25,28.41,12.59,-1.46,20.16,19.31,False
5,A,RandomForest,10.93,6.56,29.25,17.60,-4.37,22.69,20.15,False
6,B,ElasticNet,9.73,10.82,31.64,12.08,1.09,20.82,22.54,False
7,B,Ridge,10.63,9.18,34.38,12.57,-1.44,25.19,25.28,False
8,B,PLS,11.35,16.56,34.83,15.76,5.21,18.27,25.73,False
9,B,GPR,10.76,10.11,42.08,14.93,-0.65,31.97,32.98,False


## 성능 (최종 모델, 노션 Reporting 형식)

In [2]:
f = res[res.final].iloc[0]

pd.DataFrame([
    ['Train (Batch 1 CV)', f.mape_train_cv, ''],
    ['Valid (Batch 1 Hold-out)', f.mape_valid, ''],
    ['Test (Batch 2)', f.mape_test_b2, ''],
    ['Gap (Train-Valid)', f.gap_train_valid, '(+) 과적합 의심'],
    ['Gap (Valid-Test)', f.gap_valid_test, '(+) 배치 간 일반화 저하 의심'],
    ['Gap (Target-Test)', f.gap_target_test, '원논문 MAPE 9.1% 대비'],
    ['Test (Batch 3)', f.mape_test_b3, '추가'],
], columns=['구분', 'MAPE (%)', '비고'])

,구분,MAPE (%),비고
0,Train (Batch 1 CV),11.77,
1,Valid (Batch 1 Hold-out),5.84,
2,Test (Batch 2),29.63,
3,Gap (Train-Valid),-5.93,(+) 과적합 의심
4,Gap (Valid-Test),23.79,(+) 배치 간 일반화 저하 의심
5,Gap (Target-Test),20.53,원논문 MAPE 9.1% 대비
6,Test (Batch 3),18.47,추가


## 오류 분석 (Batch 2)

In [3]:
p = pd.read_csv('../results/predictions_b2.csv')
print(p.groupby('group').agg(n=('uid', 'size'), actual=('cycle_life', 'mean'), pred=('life_pred', 'mean'), mape=('ape_pct', 'mean')).round(1))
p.sort_values('ape_pct', ascending=False).head(5)

            n  actual   pred  mape
group                             
b2-legacy  30   453.0  607.2  34.7
b2-new      9   941.6  878.9  12.8


,uid,policy,group,cycle_life,life_pred,ape_pct
6,b2_c06,3.6C(9%)-5C,b2-legacy,393.0,636.4,61.9
15,b2_c15,3.6C(9%)-5C,b2-legacy,396.0,606.9,53.3
18,b2_c18,5.2C(50%)-4.25C,b2-legacy,449.0,682.9,52.1
19,b2_c19,6C(60%)-3C,b2-legacy,392.0,594.9,51.8
21,b2_c21,6C(60%)-3C,b2-legacy,408.0,594.9,45.8
